# Frankie NSFW Generator — Pony XL + IPAdapter FaceID

**One task**: generate 8 uncensored Frankie selfies for internal MVP review.

Runtime → **Run all**. Walk away ~15 min. Images appear at the bottom.

Requires: Free Colab T4 (Runtime → Change runtime type → T4 GPU).

In [ ]:
# ── 1. GPU check ──
!nvidia-smi | head -15

In [ ]:
# ── 2. Install diffusers + IPAdapter deps ──
!pip install -q diffusers==0.31.0 transformers accelerate safetensors pillow insightface onnxruntime-gpu einops
!pip install -q git+https://github.com/tencent-ailab/IP-Adapter.git || pip install -q ip_adapter

In [ ]:
# ── 3. Download Pony Diffusion v6 XL ──
import os
os.makedirs('/content/models', exist_ok=True)
!wget -q --show-progress -O /content/models/pony_v6.safetensors 'https://huggingface.co/AstraliteHeart/pony-diffusion-v6/resolve/main/v6.safetensors'
!ls -lh /content/models/

In [ ]:
# ── 4. Download IPAdapter FaceID (SDXL variant) ──
!mkdir -p /content/models/ipadapter /content/models/image_encoder
!wget -q --show-progress -O /content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin'
!wget -q --show-progress -O /content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl_lora.safetensors 'https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors'
# CLIP vision encoder for FaceID+
from huggingface_hub import snapshot_download
snapshot_download(repo_id='laion/CLIP-ViT-H-14-laion2B-s32B-b79K', local_dir='/content/models/image_encoder', allow_patterns=['*.json','*.bin','*.safetensors'])

In [ ]:
# ── 5. Load Frankie reference ──
import requests, io
from PIL import Image
FRANKIE_REF_URL = 'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png'
frankie_img = Image.open(io.BytesIO(requests.get(FRANKIE_REF_URL).content)).convert('RGB')
display(frankie_img.resize((400, 327)))
print(f'Frankie reference: {frankie_img.size}')

In [ ]:
# ── 6. Extract face embedding via InsightFace ──
import cv2, numpy as np, torch
from insightface.app import FaceAnalysis
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
frankie_cv = cv2.cvtColor(np.array(frankie_img), cv2.COLOR_RGB2BGR)
faces = face_app.get(frankie_cv)
assert len(faces) > 0, 'no face detected in reference'
face_embed = torch.from_numpy(faces[0].normed_embedding).unsqueeze(0)
face_image = frankie_img
print(f'Face embed: {face_embed.shape}')

In [ ]:
# ── 7. Load Pony XL pipeline with IPAdapter FaceID ──
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
pipe = StableDiffusionXLPipeline.from_single_file(
    '/content/models/pony_v6.safetensors',
    torch_dtype=torch.float16,
    variant='fp16',
).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_lora_weights('/content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl_lora.safetensors', adapter_name='faceid')
pipe.set_adapters(['faceid'], adapter_weights=[0.8])
from ip_adapter import IPAdapterFaceIDPlusXL
ip = IPAdapterFaceIDPlusXL(pipe, '/content/models/image_encoder', '/content/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin', 'cuda')
print('Pipeline ready')

In [ ]:
# ── 8. Generate Frankie scenes ──
# Pony prompt convention: 'score_9, score_8_up, score_7_up, source_photo, ...'
POSITIVE_PREFIX = 'score_9, score_8_up, score_7_up, source_photo, photorealistic, sharp focus, natural skin texture, freckles, dark wavy hair, gold hoop earrings, \"Frankie\" name necklace, sleeve tattoos, '
NEGATIVE = 'score_4, score_5, score_6, source_furry, source_anime, source_cartoon, source_pony, deformed, bad anatomy, extra fingers, watermark, text, signature, blurry, low quality, ugly'
SCENES = [
    'topless mirror selfie in bathroom, soft warm lighting, intimate',
    'topless in bed in morning light, smiling, candid',
    'wearing black lace lingerie, sitting on bed, looking at camera, sultry',
    'in a white t-shirt and panties, kitchen morning, holding coffee',
    'topless on a beach at sunset, ocean behind, hair wet',
    'in a sheer black robe, balcony at night, city lights behind',
    'taking a selfie in a black string bikini, pool behind, sun-kissed',
    'in a tight white tank top with no bra, gym mirror selfie, sweaty',
]
import os
os.makedirs('/content/out', exist_ok=True)
results = []
for i, scene in enumerate(SCENES, 1):
    prompt = POSITIVE_PREFIX + scene
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    img = ip.generate(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        face_image=face_image,
        faceid_embeds=face_embed,
        shortcut=True,
        s_scale=1.0,
        num_samples=1,
        width=1024, height=1024,
        num_inference_steps=30,
        guidance_scale=6.0,
        seed=42 + i,
    )[0]
    out_path = f'/content/out/frankie_{i:02d}.png'
    img.save(out_path)
    results.append((scene, img, out_path))

In [ ]:
# ── 9. Display all results in a grid ──
from IPython.display import display, Markdown
for scene, img, path in results:
    display(Markdown(f'**{scene}**'))
    display(img.resize((512, 512)))

In [ ]:
# ── 10. Zip & download all results ──
!cd /content/out && zip -q /content/frankie_nsfw_batch.zip *.png
from google.colab import files
files.download('/content/frankie_nsfw_batch.zip')